<a href="https://colab.research.google.com/github/DevD231204/DevD231204/blob/main/NLP_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2

In [1]:
!pip install transformers datasets peft evaluate accelerate
!pip install adapter-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 124.3 MB/s eta 0:00:00
  Created wheel for adapter-transformers: filename=adapter_transformers-4.0.0-py3-none-any.whl size=2695 sha256=da338539db7122760a3edeaa32487d6012111632d4604b2bce9e1786baf403e4
  Stored in directory: /root/.cache/pip/wheels/50/98/71/12adabf46fb654d220b0e11b68d827fb5d8d9b960f9cf2b432
Successfully built adapter-transformers
  Attempting uninstall: huggingface-hub
    Found existing ins

In [2]:
import torch
import time
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
dataset = load_dataset("glue", "sst2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [5]:
metric = evaluate.load("glue", "sst2")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return metric.compute(predictions=preds, references=labels)

In [6]:
training_args = TrainingArguments(
    output_dir="./results",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    logging_dir="./logs"
)

In [14]:
model_full = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
start = time.time()

trainer_full = Trainer(
    model=model_full,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

trainer_full.train()

time_full = time.time() - start

Step,Training Loss
500,0.347500
1000,0.256100
1500,0.222800
2000,0.199400
2500,0.197000
3000,0.197400
3500,0.186200
4000,0.170700
4500,0.141500
5000,0.120700


In [16]:
from peft import LoraConfig, get_peft_model

model_lora = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1
)

model_lora = get_peft_model(model_lora, config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
start = time.time()

trainer_lora = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

trainer_lora.train()

time_lora = time.time() - start

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
500,0.699600
1000,0.621200
1500,0.447800
2000,0.378700
2500,0.350900
3000,0.350600
3500,0.334900
4000,0.323300
4500,0.329500
5000,0.316300


In [7]:
import torch
import torch.nn as nn

class Adapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck)
        self.relu = nn.ReLU()
        self.up = nn.Linear(bottleneck, hidden_size)

    def forward(self, x):
        return x + self.up(self.relu(self.down(x)))

In [8]:
from transformers import AutoModel

class BertWithAdapter(nn.Module):
    def __init__(self):
        super().__init__()

        self.bert = AutoModel.from_pretrained("bert-base-uncased")

        # Freeze original BERT
        for param in self.bert.parameters():
            param.requires_grad = False

        # Add adapter per layer
        self.adapters = nn.ModuleList([
            Adapter(768) for _ in range(len(self.bert.encoder.layer))
        ])

        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask, labels=None):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden_states = outputs.last_hidden_state

        # Apply adapters AFTER full encoder (simpler & stable)
        for adapter in self.adapters:
            hidden_states = adapter(hidden_states)

        pooled = hidden_states[:, 0]
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

In [9]:
model_adapter = BertWithAdapter()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [10]:
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable params:", count_trainable(model_adapter))

Trainable params: 1191170


In [11]:
trainer_adapter = Trainer(
    model=model_adapter,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

In [12]:
import time

start = time.time()

trainer_adapter.train()

time_adapter = time.time() - start

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.506700
1000,0.416400
1500,0.395200
2000,0.386300
2500,0.396000
3000,0.391500
3500,0.389200
4000,0.374500
4500,0.385500
5000,0.382000


In [18]:
res_full = trainer_full.evaluate()
res_lora = trainer_lora.evaluate()
res_adapter = trainer_adapter.evaluate()

In [19]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

full_params = count_params(model_full)
lora_params = count_params(model_lora)
adapter_params = count_params(model_adapter)

In [20]:
print("FULL:", res_full, time_full, full_params)
print("LORA:", res_lora, time_lora, lora_params)
print("ADAPTER:", res_adapter, time_adapter, adapter_params)

FULL: {'eval_loss': 0.29548290371894836, 'eval_accuracy': 0.9277522935779816, 'eval_runtime': 14.0591, 'eval_samples_per_second': 62.593, 'eval_steps_per_second': 3.912, 'epoch': 2.0} 1529.6946804523468 (109483778, 109483778)
LORA: {'eval_runtime': 14.0274, 'eval_samples_per_second': 62.734, 'eval_steps_per_second': 3.921, 'epoch': 2.0} 778.1904304027557 (109778690, 294912)
ADAPTER: {'eval_loss': 0.3282995820045471, 'eval_accuracy': 0.8577981651376146, 'eval_runtime': 11.953, 'eval_samples_per_second': 73.621, 'eval_steps_per_second': 4.601, 'epoch': 2.0} 484.7341077327728 (110673410, 1191170)


Extensions:

In [21]:
model_freeze = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Freeze ALL layers first
for param in model_freeze.bert.parameters():
    param.requires_grad = False

# Unfreeze last 3 layers
for layer in model_freeze.bert.encoder.layer[9:]:
    for param in layer.parameters():
        param.requires_grad = True

# Always train classifier
for param in model_freeze.classifier.parameters():
    param.requires_grad = True

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
start = time.time()

trainer_freeze = Trainer(
    model=model_freeze,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

trainer_freeze.train()

time_freeze = time.time() - start

Step,Training Loss
500,0.398800
1000,0.333300
1500,0.305400
2000,0.291100
2500,0.281000
3000,0.282600
3500,0.277800
4000,0.256100
4500,0.259900
5000,0.243700


In [23]:
model_attention = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Freeze everything
for param in model_attention.parameters():
    param.requires_grad = False

# Unfreeze ONLY attention layers
for name, param in model_attention.named_parameters():
    if "attention" in name:
        param.requires_grad = True

# Also train classifier
for param in model_attention.classifier.parameters():
    param.requires_grad = True

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
start = time.time()

trainer_attention = Trainer(
    model=model_attention,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

trainer_attention.train()

time_attention = time.time() - start

Step,Training Loss
500,0.390100
1000,0.279800
1500,0.248200
2000,0.229500
2500,0.218900
3000,0.220400
3500,0.208800
4000,0.201500
4500,0.178400
5000,0.156500


In [25]:
res_freeze = trainer_freeze.evaluate()
res_attention = trainer_attention.evaluate()

In [26]:
freeze_params = count_params(model_freeze)
attention_params = count_params(model_attention)

In [27]:
print("PARTIAL FREEZE:", res_freeze, time_freeze, freeze_params)
print("ATTENTION:", res_attention, time_attention, attention_params)

PARTIAL FREEZE: {'eval_loss': 0.25907430052757263, 'eval_accuracy': 0.9162844036697247, 'eval_runtime': 2.1767, 'eval_samples_per_second': 404.275, 'eval_steps_per_second': 25.267, 'epoch': 2.0} 573.0785942077637 (109483778, 21265154)
ATTENTION: {'eval_loss': 0.2834876775741577, 'eval_accuracy': 0.9243119266055045, 'eval_runtime': 2.2452, 'eval_samples_per_second': 391.94, 'eval_steps_per_second': 24.496, 'epoch': 2.0} 1078.8386144638062 (109483778, 28368386)
